In [0]:
from pyspark.sql.functions import *

In [0]:
spark.conf.set(
    f"fs.azure.account.key.<storageaccountname>.dfs.core.windows.net",
    <"storagekey">
)

In [0]:
jobs_source = "abfss://bronze@projectstorage2612.dfs.core.windows.net/jobs_raw"
company_source = "abfss://bronze@projectstorage2612.dfs.core.windows.net/company_raw"
salary_source = "abfss://bronze@projectstorage2612.dfs.core.windows.net/salary_raw"
location_source = "abfss://bronze@projectstorage2612.dfs.core.windows.net/location_raw"
experience_source = "abfss://bronze@projectstorage2612.dfs.core.windows.net/experience_raw"
posting_source = "abfss://bronze@projectstorage2612.dfs.core.windows.net/posting_raw"

jobs_dest = "abfss://silver@projectstorage2612.dfs.core.windows.net/jobs_clean"
company_dest = "abfss://silver@projectstorage2612.dfs.core.windows.net/company_clean"
salary_dest = "abfss://silver@projectstorage2612.dfs.core.windows.net/salary_clean"
location_dest = "abfss://silver@projectstorage2612.dfs.core.windows.net/location_clean"
experience_dest = "abfss://silver@projectstorage2612.dfs.core.windows.net/experience_clean"
posting_dest = "abfss://silver@projectstorage2612.dfs.core.windows.net/posting_clean"

In [0]:
jobs_df = spark.read.format("delta").load(jobs_source)

company_df = spark.read.format("delta").load(company_source)

salary_df = spark.read.format("delta").load(salary_source)

location_df = spark.read.format("delta").load(location_source)

experience_df = spark.read.format("delta").load(experience_source)

posting_df = spark.read.format("delta").load(posting_source)

In [0]:
#duplicate rows remove
jobs_df = jobs_df.dropDuplicates()

company_df = company_df.dropDuplicates()

salary_df = salary_df.dropDuplicates()

location_df = location_df.dropDuplicates()

experience_df = experience_df.dropDuplicates()

posting_df = posting_df.dropDuplicates()

In [0]:
# handle null values
jobs_df = jobs_df.fillna({
    "title":"Unknown",
    "description":"Not Available"
})

company_df = company_df.fillna({
    "company_name":"Unknown"
})

location_df = location_df.fillna({
    "location":"Unknown"
})

experience_df = experience_df.fillna({
    "formatted_experience_level":"Unknown",
    "formatted_work_type":"Unknown",
    "work_type":"Unknown"
})

In [0]:
salary_df = salary_df.withColumn(
    "min_salary",
    col("min_salary").cast("double")
)

salary_df = salary_df.withColumn(
    "med_salary",
    col("med_salary").cast("double")
)

salary_df = salary_df.withColumn(
    "max_salary",
    col("max_salary").cast("double")
)

salary_df = salary_df.withColumn(
    "normalized_salary",
    col("normalized_salary").cast("double")
)

In [0]:
posting_df = posting_df.withColumn(
    "original_listed_time",
    to_timestamp("original_listed_time")
)

posting_df = posting_df.withColumn(
    "listed_time",
    to_timestamp("listed_time")
)

posting_df = posting_df.withColumn(
    "expiry",
    to_timestamp("expiry")
)

posting_df = posting_df.withColumn(
    "closed_time",
    to_timestamp("closed_time")
)

In [0]:
jobs_df.write.format("delta").option("overwriteSchema", "true").mode("overwrite").save(jobs_dest)

company_df.write.format("delta").option("overwriteSchema", "true").mode("overwrite").save(company_dest)

salary_df.write.format("delta").option("overwriteSchema", "true").mode("overwrite").save(salary_dest)

location_df.write.format("delta").option("overwriteSchema", "true").mode("overwrite").save(location_dest)

experience_df.write.format("delta").option("overwriteSchema", "true").mode("overwrite").save(experience_dest)

posting_df.write.format("delta").option("overwriteSchema", "true").mode("overwrite").save(posting_dest)

In [0]:
spark.sql(f"OPTIMIZE delta.`{jobs_dest}`")
spark.sql(f"OPTIMIZE delta.`{company_dest}`")
spark.sql(f"OPTIMIZE delta.`{salary_dest}`")
spark.sql(f"OPTIMIZE delta.`{location_dest}`")
spark.sql(f"OPTIMIZE delta.`{experience_dest}`")
spark.sql(f"OPTIMIZE delta.`{posting_dest}`")

DataFrame[path: string, metrics: struct<numFilesAdded:bigint,numFilesRemoved:bigint,filesAdded:struct<min:bigint,max:bigint,avg:double,totalFiles:bigint,totalSize:bigint>,filesRemoved:struct<min:bigint,max:bigint,avg:double,totalFiles:bigint,totalSize:bigint>,partitionsOptimized:bigint,zOrderStats:struct<strategyName:string,inputCubeFiles:struct<num:bigint,size:bigint>,inputOtherFiles:struct<num:bigint,size:bigint>,inputNumCubes:bigint,mergedFiles:struct<num:bigint,size:bigint>,numOutputCubes:bigint,mergedNumCubes:bigint>,clusteringStats:struct<inputZCubeFiles:struct<numFiles:bigint,size:bigint>,inputOtherFiles:struct<numFiles:bigint,size:bigint>,inputNumZCubes:bigint,mergedFiles:struct<numFiles:bigint,size:bigint>,numOutputZCubes:bigint>,numBins:bigint,numBatches:bigint,totalConsideredFiles:bigint,totalFilesSkipped:bigint,preserveInsertionOrder:boolean,numFilesSkippedToReduceWriteAmplification:bigint,numBytesSkippedToReduceWriteAmplification:bigint,startTimeMs:bigint,endTimeMs:bigint,

In [0]:
spark.sql(f"VACUUM delta.`{jobs_dest}` RETAIN 168 HOURS")
spark.sql(f"VACUUM delta.`{company_dest}` RETAIN 168 HOURS")
spark.sql(f"VACUUM delta.`{salary_dest}` RETAIN 168 HOURS")
spark.sql(f"VACUUM delta.`{location_dest}` RETAIN 168 HOURS")
spark.sql(f"VACUUM delta.`{experience_dest}` RETAIN 168 HOURS")
spark.sql(f"VACUUM delta.`{posting_dest}` RETAIN 168 HOURS")

com.databricks.backend.common.rpc.CommandCancelledException
	at com.databricks.spark.chauffeur.SequenceExecutionState.$anonfun$cancel$5(SequenceExecutionState.scala:132)
	at scala.Option.getOrElse(Option.scala:201)
	at com.databricks.spark.chauffeur.SequenceExecutionState.$anonfun$cancel$3(SequenceExecutionState.scala:132)
	at com.databricks.spark.chauffeur.SequenceExecutionState.$anonfun$cancel$3$adapted(SequenceExecutionState.scala:129)
	at scala.collection.immutable.Range.foreach(Range.scala:190)
	at com.databricks.spark.chauffeur.SequenceExecutionState.cancel(SequenceExecutionState.scala:129)
	at com.databricks.spark.chauffeur.ExecContextState.cancelRunningSequence(ExecContextState.scala:721)
	at com.databricks.spark.chauffeur.ExecContextState.$anonfun$cancel$1(ExecContextState.scala:441)
	at scala.Option.getOrElse(Option.scala:201)
	at com.databricks.spark.chauffeur.ExecContextState.cancel(ExecContextState.scala:441)
	at com.databricks.spark.chauffeur.ExecutionContextManagerV1.can

In [0]:
print("Silver Layer Created Successfully")